# Transaction Foundation Model on Ray — Part 3: Tokenize with NVIDIA's tokenizer

<div align="left">
  <a target="_blank" href="https://console.anyscale.com/template-preview/fintech_transaction_fm"><img src="https://img.shields.io/badge/🚀 Run_on-Anyscale-9hf"></a>&nbsp;
  <a href="https://github.com/anyscale/templates/tree/main/templates/fintech_transaction_fm" role="button"><img src="https://img.shields.io/static/v1?label=&message=View%20On%20GitHub&color=586069&logo=github&labelColor=2f363d"></a>&nbsp;
</div>

**⏱️ Time to complete**: ~5 min (full: longer — encodes the whole train split)

---

In Part 2 we built the temporal split. This notebook turns its train part into what the model actually reads: token ids. Every transaction becomes 12 tokens from a fixed vocabulary, every card's history becomes one token stream, and the streams are cut into fixed-length sequences — the pretraining corpus for Part 4. The tokenizer is NVIDIA's own, vendored verbatim into `src/nvidia_tokenizer/`, because tokenization defines what the model can learn — a reimplementation would make every later comparison suspect. The build runs distributed, entirely on CPU workers.

In [1]:
import sys, os, json

DEMO_ROOT = os.path.abspath(os.getcwd())
if DEMO_ROOT not in sys.path:
    sys.path.insert(0, DEMO_ROOT)

import numpy as np
import pandas as pd
import logging

import ray
ray.init(ignore_reinit_error=True, runtime_env={"working_dir": DEMO_ROOT},
         logging_level=logging.ERROR)

from src.paths import artifact_paths, get_demo_base_dir
from src.scale_config import load_scale

SCALE = "mini"                                        # same knob as Part 2; full = every card
cfg = load_scale(SCALE)
paths = artifact_paths(get_demo_base_dir(), SCALE)

/home/ray/anaconda3/lib/python3.12/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


## From transactions to tokens

A language model never sees words — it sees ids from a fixed vocabulary, and everything it learns is expressed over those ids. The same holds here, with one difference: a transaction isn't text, so there is nothing to split into subwords. Instead, every field maps to its own family of tokens. Two fields have values that can't be enumerated, so they get bucketed or hashed until they can:

- **Amount** is continuous, so it's bucketed at fixed dollar thresholds (\$0/10/50/100/500/1K/5K), one token per bucket. Part 2 showed amounts spanning orders of magnitude, which is why the thresholds sit roughly one per factor of ten rather than at even spacing.
- **Merchant** is open-ended (~91K distinct names), so it's hashed into 2,000 buckets. Hashing bounds the vocabulary; collisions are the accepted cost.
- **MCC appears twice** — the raw code (`MCC_5411`) plus a coarse industry category derived from it (`CAT_RETAIL`) — so the model can generalize across related merchants even when a specific code is rare.
- **Time is explicit**: hour, day-of-week, and month tokens on every transaction. Part 2's burstiness is the motivation — position in the sequence says nothing about elapsed time. (NVIDIA also ships a gap-since-last-transaction tokenizer; the blueprint leaves it off, and so do we.)
- The rest are small categoricals mapped directly: card index, chip/swipe/online channel, ZIP3 region, state, customer id.

Twelve tokens per transaction, all drawn from one shared vocabulary of 6,251 ids. A card's history becomes a single stream — `<bos> [txn₁] <sep> [txn₂] <sep> … <eos>` — so with the `<sep>` each transaction occupies ~13 positions. The stream is cut into windows of `seq_len` tokens: 4096 at full scale (~315 transactions of context per window), 256 at `mini`. Those windows are the corpus Part 4 trains on by next-token prediction.

The bucketing is lossy on purpose — a \$12 and a \$47 purchase share a token — and that loss is part of why the raw fields still carry independent signal when Part 6 combines them with embeddings.

## Why this stage runs on CPUs

NVIDIA's tokenizer is written in cuDF, so in the blueprint even data preparation occupies a GPU. That is the wrong hardware for the job: tokenization is string formatting and dictionary lookups, and a GPU is the most expensive way to buy those. This template runs a pandas mirror of their tokenizer (`src/nvtokenize_cpu.py`) on ordinary CPU workers and keeps the cuDF original vendored as the reference.

The mirror is not approximately right — it is byte-identical, and that is checked rather than assumed. The subtle part is the merchant hash: cuDF's `hash_values` is murmur3 combined the Boost way, and the mirror reproduces it bit-for-bit. `scripts/verify_cpu_tokenizer.py` compares all ~91K merchant hashes, all 12 token columns, and the full vocabulary against the GPU implementation; `scripts/verify_distributed_corpus.py` then checks the assembled corpus against a single-GPU reference build. Byte-identical inputs are what make the model comparison in Part 6 meaningful.

## The corpus build as a Ray Data job

No card's tokens depend on any other card's, which gives the job its shape: `groupby → map_groups`. Ray Data hash-shuffles the train rows so each card's transactions land together, then runs one card at a time across the CPU workers — restore time order, derive the 12 token strings per transaction, pack and encode to `seq_len` ids. The shuffle is the real distributed work here; the per-card function is ordinary pandas.

The result is three arrays the pretrainer reads directly — `ids.npy` (`(n_sequences, seq_len)` int32, right-padded), `attn.npy` (the attention mask), and `vocab.json` — assembled in a deterministic (user, card, chunk) order so re-runs and the identity checks are stable. Re-runs reuse the cached corpus.

In [2]:
from src.nvsplit import train_parquet_files
from src.nvcorpus import tokenize_card_group, fresh_seqs_dir, assemble_corpus

tk = cfg["tokenize"]
seq_len = tk["seq_len"]                        # tokens per pretrain sequence (full 4096 = NVIDIA)
chunk = max(1, seq_len // 13)                  # ~13 tokens/txn → transactions packed per sequence
max_seq = tk.get("max_pretrain_windows")       # cap #sequences (mini/CI); None = every sequence

# Shuffle into per-card groups, tokenize each card on a CPU worker, assemble.
if not os.path.exists(os.path.join(paths["nvcorpus"], "ids.npy")):
    seqs_dir = fresh_seqs_dir(paths["nvcorpus"])
    ray.data.read_parquet(train_parquet_files(paths["nvsplit"])) \
        .groupby(["User", "Card"]) \
        .map_groups(lambda g: tokenize_card_group(g, seq_len, chunk),
                    batch_format="pandas") \
        .write_parquet(seqs_dir)
    meta = assemble_corpus(seqs_dir, paths["nvcorpus"], seq_len, max_seq)
    print(json.dumps(meta, indent=2))
else:
    print("corpus already built at", paths["nvcorpus"])

corpus already built at /mnt/cluster_storage/transaction-fm/nvcorpus/mini/


## The tokenized corpus

Let's look at what we built: the number of sequences and the fraction of real (non-pad) tokens, then decode the first sequence back to readable tokens using the same tokenizer. Each transaction's field tokens appear in order, framed by `<bos>` / `<sep>` / `<eos>`; merchant is hashed, so it reads as a bucket id. (Importing the vendored tokenizer pulls in cuDF, which warns that it finds no GPU — expected: everything here runs on CPU.)

In [3]:
from src.nvidia_tokenizer import FinancialTabularTokenizer

ids = np.load(os.path.join(paths["nvcorpus"], "ids.npy"), mmap_mode="r")
attn = np.load(os.path.join(paths["nvcorpus"], "attn.npy"), mmap_mode="r")
with open(os.path.join(paths["nvcorpus"], "vocab.json")) as f:
    vinfo = json.load(f)

print(f"{ids.shape[0]:,} pretrain sequences × {ids.shape[1]} tokens  "
      f"(vocab {vinfo['vocab_size']:,}, pad id {vinfo['pad']})")
print(f"real-token fraction: {float(np.asarray(attn).mean()):.3f}\n")

# Decode sequence 0 back to readable tokens (drop right-padding).
tok = FinancialTabularTokenizer(merchant_hash_size=2000, category_hierarchy=True,
                                temporal_encoding=True)
row = np.asarray(ids[0])
real = [int(t) for t in row if int(t) != vinfo["pad"]]
decoded = tok.decode(real).split()
print(f"sequence 0 — {len(real)} real tokens; first ~2 transactions:")
print("  " + " ".join(decoded[:32]))

/home/ray/anaconda3/lib/python3.12/site-packages/cudf/utils/gpu_utils.py:162: UserWarning: No NVIDIA GPU detected
  warnings.warn("No NVIDIA GPU detected")


8 pretrain sequences × 256 tokens  (vocab 6,251, pad id 0)
real-token fraction: 0.969

sequence 0 — 248 real tokens; first ~2 transactions:
  <bos> AMT_3 MERCH_667 CAT_RETAIL MCC_5300 HOUR_06 DOW_6 MONTH_09 CARD_0 CHIP_SWIPE ZIP3_917 STATE_CA CUST_0 <sep> AMT_1 MERCH_869 CAT_RETAIL MCC_5411 HOUR_06 DOW_6 MONTH_09 CARD_0 CHIP_SWIPE ZIP3_917 STATE_CA CUST_0 <sep> AMT_3 MERCH_869 CAT_RETAIL MCC_5411 HOUR_06


The vocabulary is fixed at **6,251 tokens** and known before any data is seen, so the model's embedding table can be sized up front. Special tokens (`<pad>`/`<bos>`/`<eos>`/`<sep>`/`<unk>`) come first, then each field owns a disjoint block of ids (hashed merchant buckets, category hierarchy, temporal fields, amount bins).

In [4]:
vocab = tok.vocab   # same tokenizer instance as the decode above
specials = [t for t in ("<pad>", "<bos>", "<eos>", "<sep>", "<unk>") if t in vocab]
print(f"shared vocabulary: {tok.vocab_size:,} tokens  ({len(specials)} special + per-field blocks)")
print(f"specials: {specials}\n")

# Group the vocab tokens by their field prefix to show the per-field block sizes.
from collections import Counter
prefixes = Counter(t.split("_")[0] for t in vocab if t not in specials and "_" in t)
print("per-field block size in the shared vocab:")
for name, size in sorted(prefixes.items(), key=lambda kv: -kv[1]):
    print(f"  {name:<12} {size:>6,}")

shared vocabulary: 6,251 tokens  (5 special + per-field blocks)
specials: ['<pad>', '<bos>', '<eos>', '<sep>', '<unk>']

per-field block size in the shared vocab:
  CUST          3,000
  MERCH         2,000
  ZIP3          1,000
  MCC             110
  STATE            58
  HOUR             24
  CAT              14
  MONTH            12
  CARD             10
  AMT               7
  DOW               7
  CHIP              4


## Scaling factors

Three things grow when the data does, and they scale differently. The shuffle moves every train row once, so its cost is linear in transactions — that is the stage's distributed work, and more CPU workers absorb it. The parallelism grain is cards: ~6,100 here, millions at a real institution, so the job gets more parallel as it gets bigger, not less. And the corpus itself is linear in transactions — at full scale, 64,335 sequences × 4096 tokens is about 1 GB of `ids.npy` — with `seq_len` and `max_pretrain_windows` as the two knobs that trade context length and corpus size against training cost. The code names none of these sizes; `mini` and `full` differ only in config.

## Takeaways

The train split is now a pretraining corpus: 12 tokens per transaction from NVIDIA's vendored tokenizer, one stream per card, packed into `seq_len` windows. The stage ran entirely on CPU workers as a `groupby → map_groups` job, using a pandas mirror verified byte-identical to the cuDF original — merchant hashes, token columns, vocabulary, and the assembled corpus all checked against the single-GPU reference. The result is the corpus NVIDIA's code produces, built without a GPU.

---

## Next

**Part 4 — Pretrain with Ray Train**: next-token prediction over these sequences with a Llama causal decoder — plain PyTorch wrapped by Ray Train for distributed data parallelism and checkpointing, the same code from one worker to eight GPUs.